### 1. Data Preparation & Exploration
Kita akan membuat dataset sintetis untuk mensimulasikan data peminjam.

In [10]:
import sys
print(sys.executable)

d:\FUTURES\GitHub\credit-risk-modeling\.venv\Scripts\python.exe


In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from src.risk import calculate_expected_loss

# Membaca data dari file yang diupload
from src.data_loader import load_loan_data
df = load_loan_data()

# Menampilkan info kolom untuk memastikan nama fitur
print("Kolom yang tersedia:", df.columns.tolist())

# Kita asumsikan kolom target default sudah ada di dataset
# Jika nama kolom berbeda (misal: 'loan_status' atau 'is_default'), sesuaikan di bawah
display(df.head())
print(f"Total data: {len(df)} baris")

Kolom yang tersedia: ['customer_id', 'credit_lines_outstanding', 'loan_amt_outstanding', 'total_debt_outstanding', 'income', 'years_employed', 'fico_score', 'default']


,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0


Total data: 10000 baris


### 2. Model Development
Kita akan melatih model Logistic Regression untuk memprediksi probabilitas default.

In [12]:
# Pastikan fitur (X) sesuai dengan kolom yang ada di CSV
# Misalnya: ['income', 'loan_amount', 'credit_score']
# Sesuaikan list di bawah ini dengan nama kolom asli di file lo
features = [col for col in df.columns if col != 'default']
X = df[features]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluasi model
y_pred_prob = model.predict_proba(X_test)[:, 1]
print(f"Model AUC Score: {roc_auc_score(y_test, y_pred_prob):.4f}")

Model AUC Score: 0.9985


### 3. Expected Loss Function
Rumus Expected Loss (EL) adalah:
`EL = PD * LGD * EAD`
dimana:
*   **PD (Probability of Default)**: Diambil dari model.
*   **LGD (Loss Given Default)**: 1 - Recovery Rate (1 - 0.10 = 0.90).
*   **EAD (Exposure at Default)**: Total pinjaman yang outstanding.

In [13]:

# Contoh penggunaan fungsi dengan data sesuai format CSV
sample_loan = calculate_expected_loss(
    model=model,
    features=features,
    customer_id=12345,
    credit_lines_outstanding=2,
    loan_amt_outstanding=5000,
    total_debt_outstanding=10000,
    income=50000,
    years_employed=5,
    fico_score=650
)
print(f"Hasil Kalkulasi: {sample_loan}")

Hasil Kalkulasi: {'probability_of_default': np.float64(0.5735), 'expected_loss': np.float64(2580.91)}


he risk manager has collected data on the loan borrowers. The data is in tabular format, with each row providing details of the borrower, including their income, total loans outstanding, and a few other metrics. There is also a column indicating if the borrower has previously defaulted on a loan. You must use this data to build a model that, given details for any loan described above, will predict the probability that the borrower will default (also known as PD: the probability of default). Use the provided data to train a function that will estimate the probability of default for a borrower. Assuming a recovery rate of 10%, this can be used to give the expected loss on a loan.

You should produce a function that can take in the properties of a loan and output the expected loss.
You can explore any technique ranging from a simple regression or a decision tree to something more advanced. You can also use multiple methods and provide a comparative analysis.

1. Dapet Probability of Default (PD) = 0.0711
Angka ini keluar dari Model Logistic Regression yang udah kita latih tadi.

Pas lo panggil fungsi calculate_expected_loss, data input (fico_score 650, income 50rb, dll) dimasukin ke model.
Model ngelihat pola dari 10.000 data di CSV lo. Dia nemu kalau orang dengan profil kayak gitu punya kemungkinan 7.11% buat gagal bayar (default) berdasarkan sejarah data yang ada.

2. Dapet Expected Loss (EL) = 319.85
Nah, kalau PD-nya udah dapet, tinggal masukin ke rumus standar perbankan yang gue tulis di kode:

Rumus: EL = PD × LGD × EAD

PD (Probability of Default): 0.0711 (7.11%)
LGD (Loss Given Default): Ini adalah sisa kerugian setelah dipotong recovery rate. Karena recovery rate lo 10% (0.10), maka kerugian bersihnya adalah 90% atau 0.90.
EAD (Exposure at Default): Total pinjaman yang lagi jalan, yaitu 5,000.

Hitungannya: EL = 0.0711 × 0.90 × 5,000 EL = 0.06399 × 5,000 EL = 319.95 (ada sedikit perbedaan desimal karena pembulatan di sistem).

Jadi, secara statistik, dari pinjaman 5.000 itu, bank harus siap-siap rugi sekitar $319.85 karena adanya risiko gagal bayar tadi.